# DeepRawNet — Training on SceneFake Dataset

**Model:** `DeepRawNet` (SincConv + ResidualBlocks + GRU)  
**Dataset:** [SceneFake on Kaggle](https://www.kaggle.com/datasets/mohammedabdeldayem/scenefake)  
**Label convention:** `0 = real (bonafide)`, `1 = fake (spoof)`  
**Output:** LogSoftmax → NLLLoss  
**Saved model:** `deeprawnet_scenefake.pth`

## Step 1 — Install dependencies

In [1]:
!pip install -q torchaudio kaggle

## Step 2 — Enter Kaggle API credentials and download the dataset

Run this cell. It will ask for your **Kaggle username** and **API key**.  
Find these at: [kaggle.com](https://www.kaggle.com) → Your Profile → Settings → API → **Create New Token**  
Your username and key are inside the `kaggle.json` that gets downloaded.

In [2]:
import os
import json
from getpass import getpass

# Prompt for credentials (getpass hides the key as you type)
kaggle_username = input('Enter your Kaggle username: ').strip()
kaggle_key      = getpass('Enter your Kaggle API key: ').strip()

# Write kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('Credentials saved. Downloading dataset...')

# Download SceneFake dataset
!kaggle datasets download -d mohammedabdeldayem/scenefake -p /content/scenefake --unzip

print('\nDataset downloaded. Folder structure:')

!find /content/scenefake -maxdepth 3 -type d

Enter your Kaggle username: atharvapandey27
Enter your Kaggle API key: ··········
Credentials saved. Downloading dataset...
Dataset URL: https://www.kaggle.com/datasets/mohammedabdeldayem/scenefake
License(s): CC-BY-NC-SA-4.0
100% 5.37G/5.37G [01:07<00:00, 85.3MB/s]


Dataset downloaded. Folder structure:
/content/scenefake
/content/scenefake/train
/content/scenefake/train/real
/content/scenefake/train/fake
/content/scenefake/eval
/content/scenefake/eval/real
/content/scenefake/eval/fake
/content/scenefake/dev
/content/scenefake/dev/real
/content/scenefake/dev/fake


## Step 3 — Set dataset root path

After the cell above prints the folder structure, confirm the path below is correct.  
It should point to the folder that contains `train/`, `dev/`, `eval/`.

In [3]:
DATABASE_PATH = '/content/scenefake'  # adjust if the unzip created a subfolder

## Step 4 — Upload deeprawnet.py

In [4]:
from google.colab import files
files.upload()  # upload deeprawnet.py

Saving deeprawnet (1).py to deeprawnet (1) (1).py


{'deeprawnet (1) (1).py': b'"""\r\nDeepRawNet: Empowering Deepfake Audio Detection through Dynamic Enhancements\r\nPaper: Alharbi et al. (2026), PeerJ Comput. Sci., DOI 10.7717/peerj-cs.3670\r\n\r\nArchitecture based on RawNet2 with three key innovations:\r\n  1. PReLU activation (learnable negative slope) in residual blocks \xe2\x86\x92 replaces LeakyReLU\r\n  2. Transpose Convolution in residual blocks \xe2\x86\x92 replaces standard Conv (addresses downsampling)\r\n  3. LogSoftmax in the output layer \xe2\x86\x92 replaces Softmax (numerical stability)\r\n\r\nAdditional change from paper:\r\n  - Increased negative slope of LeakyReLU in Fixed Sinc filters to 0.5\r\n\r\nOverfitting fixes for small datasets (e.g. 500 samples):\r\n  - Dropout added inside ResidualBlock (default 0.3)\r\n  - Dropout added in GRU (default 0.3)\r\n  - Dropout added before FC layer (default 0.5)\r\n  - L2 regularization via weight_decay in optimizer (already present)\r\n  - All dropout rates controllable via d

## Step 5 — Imports

In [6]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from deeprawnet import DeepRawNet, get_optimizer, get_loss

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

Using device: cuda


## Step 6 — Dataset class

In [7]:
TARGET_SR  = 16000
MAX_SAMPLES = TARGET_SR * 4  # 4 seconds of audio

class SceneFakeDataset(Dataset):
    """
    Reads .wav files from:
        database_path/split/real/*.wav  → label 0
        database_path/split/fake/*.wav  → label 1

    Resamples to 16 kHz, converts to mono, pads/trims to MAX_SAMPLES.
    DeepRawNet expects input shape: (batch, 1, num_samples)
    """
    def __init__(self, database_path, split):
        self.samples = []  # list of (filepath, label)
        for label_name, label in [('real', 0), ('fake', 1)]:
            folder = os.path.join(database_path, split, label_name)
            if not os.path.isdir(folder):
                raise FileNotFoundError(f'Expected folder not found: {folder}')
            for fname in os.listdir(folder):
                if fname.endswith('.wav'):
                    self.samples.append((os.path.join(folder, fname), label))
        print(f'[{split}] Loaded {len(self.samples)} files')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        waveform, sr = torchaudio.load(path)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if sr != TARGET_SR:
            waveform = T.Resample(sr, TARGET_SR)(waveform)

        # Pad or trim to MAX_SAMPLES
        n = waveform.shape[1]
        if n < MAX_SAMPLES:
            waveform = torch.nn.functional.pad(waveform, (0, MAX_SAMPLES - n))
        else:
            waveform = waveform[:, :MAX_SAMPLES]

        # DeepRawNet expects (batch, 1, samples) — keep channel dim
        # waveform shape here: (1, MAX_SAMPLES) — correct

        return waveform, torch.tensor(label, dtype=torch.long)

## Step 7 — Hyperparameters

In [8]:
BATCH_SIZE = 8       # reduce to 4 if you get OOM errors
NUM_EPOCHS = 10
LR         = 1e-4
WEIGHT_DECAY = 1e-4  # as per paper
SAVE_PATH  = 'deeprawnet_scenefake.pth'

## Step 8 — Build dataloaders

In [9]:
train_dataset = SceneFakeDataset(DATABASE_PATH, 'train')
dev_dataset   = SceneFakeDataset(DATABASE_PATH, 'dev')
eval_dataset  = SceneFakeDataset(DATABASE_PATH, 'eval')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
eval_loader  = DataLoader(eval_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

[train] Loaded 13185 files
[dev] Loaded 12843 files
[eval] Loaded 32746 files


## Step 9 — Model, optimizer, loss

Uses `get_optimizer()` and `get_loss()` directly from `deeprawnet.py` — no changes.

In [10]:
model     = DeepRawNet(num_classes=2, sample_rate=16000, dropout_rate=0.3).to(DEVICE)
optimizer = get_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)
criterion = get_loss()  # NLLLoss — pairs with LogSoftmax output

print(model)

DeepRawNet(
  (sinc_block): SincConvBlock(
    (sinc): SincConv()
    (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (activation): LeakyReLU(negative_slope=0.5)
  )
  (res_blocks_128): Sequential(
    (0): ResidualBlock(
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (prelu1): PReLU(num_parameters=1)
      (tconv1): ConvTranspose1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (prelu2): PReLU(num_parameters=1)
      (tconv2): ConvTranspose1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (dropout): Dropout(p=0.3, inplace=False)
      (pool): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
      (fms): FMS(
        (fc): Linear(in_features=128, out_features=1

## Step 10 — Training loop

In [18]:
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import numpy as np

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    all_scores = []
    all_labels = []

    with torch.no_grad():
        for waveforms, labels in loader:
            waveforms = waveforms.to(DEVICE)  # (B, 1, samples)
            labels    = labels.to(DEVICE)      # (B,)

            log_probs = model(waveforms)        # (B, 2)

            loss = criterion(log_probs, labels)

            total_loss += loss.item() * len(labels)

            predicted = log_probs.argmax(dim=1)

            correct += (predicted == labels).sum().item()

            total += len(labels)

            # Scores for EER (probability of class 1)
            probs = torch.softmax(log_probs, dim=1)[:, 1]

            all_scores.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Compute EER
    fpr, tpr, thresholds = roc_curve(all_labels, all_scores, pos_label=1)

    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    eer = eer * 100

    return total_loss / total, correct / total, eer


best_dev_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0

    for batch_idx, (waveforms, labels) in enumerate(train_loader):
        waveforms = waveforms.to(DEVICE)  # (B, 1, samples)
        labels    = labels.to(DEVICE)      # (B,)

        optimizer.zero_grad()

        log_probs = model(waveforms)        # (B, 2)

        loss = criterion(log_probs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * len(labels)

        predicted = log_probs.argmax(dim=1)

        running_correct += (predicted == labels).sum().item()

        running_total += len(labels)

        if (batch_idx + 1) % 100 == 0:
            print(f'  Epoch {epoch} | Batch {batch_idx+1}/{len(train_loader)} '
                  f'| Loss: {running_loss/running_total:.4f} '
                  f'| Acc: {running_correct/running_total:.4f}')

    train_loss = running_loss / running_total
    train_acc  = running_correct / running_total

    dev_loss, dev_acc, dev_eer = evaluate(model, dev_loader, criterion)

    print(f'\nEpoch {epoch}/{NUM_EPOCHS} Summary:')
    print(f'  Train  — Loss: {train_loss:.4f} | Acc: {train_acc:.4f}')
    print(f'  Dev    — Loss: {dev_loss:.4f} | Acc: {dev_acc:.4f} | EER: {dev_eer:.2f}%')

    # Save best model based on dev loss
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss

        torch.save(model.state_dict(), SAVE_PATH)

        print(f'  ✓ Best model saved to {SAVE_PATH}\n')

    else:
        print()

  Epoch 1 | Batch 100/1649 | Loss: 0.0390 | Acc: 0.9862
  Epoch 1 | Batch 200/1649 | Loss: 0.0759 | Acc: 0.9775
  Epoch 1 | Batch 300/1649 | Loss: 0.0749 | Acc: 0.9775
  Epoch 1 | Batch 400/1649 | Loss: 0.0700 | Acc: 0.9784
  Epoch 1 | Batch 500/1649 | Loss: 0.0624 | Acc: 0.9808
  Epoch 1 | Batch 600/1649 | Loss: 0.0588 | Acc: 0.9821
  Epoch 1 | Batch 700/1649 | Loss: 0.0559 | Acc: 0.9825
  Epoch 1 | Batch 800/1649 | Loss: 0.0589 | Acc: 0.9812
  Epoch 1 | Batch 900/1649 | Loss: 0.0611 | Acc: 0.9810
  Epoch 1 | Batch 1000/1649 | Loss: 0.0586 | Acc: 0.9816
  Epoch 1 | Batch 1100/1649 | Loss: 0.0576 | Acc: 0.9817
  Epoch 1 | Batch 1200/1649 | Loss: 0.0583 | Acc: 0.9809
  Epoch 1 | Batch 1300/1649 | Loss: 0.0565 | Acc: 0.9810
  Epoch 1 | Batch 1400/1649 | Loss: 0.0570 | Acc: 0.9803
  Epoch 1 | Batch 1500/1649 | Loss: 0.0539 | Acc: 0.9812
  Epoch 1 | Batch 1600/1649 | Loss: 0.0536 | Acc: 0.9816

Epoch 1/10 Summary:
  Train  — Loss: 0.0525 | Acc: 0.9819
  Dev    — Loss: 2.6211 | Acc: 0.4299 

## Step 11 — Evaluate on eval set using best saved model

In [19]:
from sklearn.metrics import accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from tqdm import tqdm
import numpy as np

# Load best model
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

preds   = []
labels  = []
scores  = []

eval_bar = tqdm(eval_loader, desc='Evaluating', unit='batch', ncols=80)

with torch.no_grad():
    for x, y in eval_bar:

        x = x.to(DEVICE)

        output = model(x)

        # For BCEWithLogitsLoss
        if output.shape[1] == 1:
            prob = torch.sigmoid(output).squeeze(1)

            pred = (prob > 0.5).long()

            scores.extend(prob.cpu().numpy())

        # For CrossEntropyLoss
        else:
            prob = torch.softmax(output, dim=1)[:, 1]

            pred = torch.argmax(output, dim=1)

            scores.extend(prob.cpu().numpy())

        preds.extend(pred.cpu().tolist())
        labels.extend(y.tolist())

# Accuracy
accuracy   = accuracy_score(labels, preds) * 100
error_rate = 100 - accuracy

# EER Calculation
fpr, tpr, thresholds = roc_curve(labels, scores)

eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
eer = eer * 100

print('=' * 50)
print('EVALUATION RESULTS')
print('=' * 50)
print(f'Total Samples : {len(labels)}')
print(f'Accuracy      : {accuracy:.2f}%')
print(f'Error Rate    : {error_rate:.2f}%')
print(f'EER           : {eer:.2f}%')
print('=' * 50)

Evaluating: 100%|████████████████████████| 4094/4094 [06:18<00:00, 10.82batch/s]

EVALUATION RESULTS
Total Samples : 32746
Accuracy      : 85.56%
Error Rate    : 14.44%
EER           : 19.04%


## Step 12 — Download the saved .pth file to your machine

In [14]:
from google.colab import files
files.download(SAVE_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>